# E-Commerce Behaviour Pipeline
## Big Data Project — Group A7

Multi-stage distributed data pipeline processing **~14.6 GB** of e-commerce event logs using **Dask** and **Apache Spark**.

| | |
|---|---|
| **Dataset** | [Kaggle — eCommerce behavior data from multi-category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) |
| **Files** | `2019-Oct.csv` (5.6 GB) · `2019-Nov.csv` (9.0 GB) |
| **Schema** | `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |

### Pipeline Stages
1. **Ingestion & Validation** — load CSVs, schema enforcement, null / range checks, cleaning
2. **Dask Processing** — `filter`, `aggregate`, `pivot`, `assign`, advanced `groupby` (5 analyses)
3. **Spark Processing** — same questions via PySpark + window function, reads **raw CSV directly**
4. **Storage** — intermediate + final results written as **Parquet**
5. **Framework Comparison** — timing and ergonomics summary

In [1]:
import os
os.environ['TEST_MODE'] = 'False'


In [2]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')   # make src/ importable from notebooks/

import dask
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from pathlib import Path
import pandas as pd

from src.config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, LOGS_DIR,
    RAW_OCT, RAW_NOV,
    PARQUET_VALIDATED, RESULTS_DIR,
    VALID_EVENT_TYPES, DTYPES,
)
from src.logger     import StructuredLogger
from src.validation import validate_schema, data_quality_report, clean_data
from src.ingestion  import load_csv, load_csvs
from src.transformations_dask import (
    compute_revenue_by_category,
    compute_conversion_funnel,
    compute_hourly_activity,
    compute_session_stats,
    compute_top_brands,
)
from src.storage import save_parquet_dask, save_parquet_pandas, load_parquet

try:
    from pyspark.sql import SparkSession
    from src.transformations_spark import (
        get_spark_session, load_csv_spark,
        compute_revenue_by_category_spark,
        compute_conversion_funnel_spark,
        compute_window_rank_spark,
        compute_hourly_activity_spark,
    )
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print('PySpark not found — Spark stage will be skipped.')

log = StructuredLogger('pipeline')
timings: dict[str, float] = {}

for d in [OUTPUT_DIR, LOGS_DIR, PARQUET_VALIDATED, RESULTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Spark        : {SPARK_AVAILABLE}')

{"ts": "2026-03-16T10:46:50.996023+00:00", "level": "INFO", "event": "spark_env_fix", "action": "unsetting_spark_home", "original": "/home/albin/Skola/Big Data/Lab1/spark"}
{"ts": "2026-03-16T10:46:50.996695+00:00", "level": "INFO", "event": "spark_env_fix", "action": "setting_java_home", "new_path": "/usr/lib/jvm/java-21-openjdk-amd64"}


Project root : /home/albin/Skola/Big Data/Project/github/Group-A7-Project
Data dir     : /home/albin/Skola/Big Data/Project/github/Group-A7-Project/data
Output dir   : /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output
Spark        : True


---
## Stage 1 — Data Ingestion & Validation

**Operations:** load two CSVs → schema check → null / value-range assertions → row-level cleaning  
**Error handling:** `on_bad_lines='skip'` drops malformed CSV rows; retry logic handles transient I/O failures; `RuntimeError` raised on schema mismatch.

> **Note:** `load_csvs()` passes both file paths directly to `dd.read_csv` as a list — Dask handles this natively without any concat or merge pass. All computation is **lazy** until `save_parquet_dask()` triggers the first full read.

In [3]:
log.info('pipeline_started')

# ── Load both CSVs in one lazy read (no merge/concat pass) ───────────────────
with log.timer('ingestion') as t:
    raw_ddf = load_csvs([RAW_OCT, RAW_NOV])
timings['ingestion'] = t.elapsed

print(f'Partitions : {raw_ddf.npartitions}')
print(f'Columns    : {list(raw_ddf.columns)}')

# ── Schema validation (fast — no compute triggered) ──────────────────────────
schema = validate_schema(raw_ddf)
print(f'\nSchema valid : {schema["valid"]}')
if not schema['valid']:
    raise RuntimeError(f'Missing columns: {schema["missing"]}')

raw_ddf.head(3)

{"ts": "2026-03-16T10:46:51.003523+00:00", "level": "INFO", "event": "pipeline_started"}
{"ts": "2026-03-16T10:46:51.004457+00:00", "level": "INFO", "event": "ingestion_started"}
{"ts": "2026-03-16T10:46:51.004813+00:00", "level": "INFO", "event": "load_csvs_started", "files": ["/home/albin/Skola/Big Data/Project/github/Group-A7-Project/data/2019-Oct.csv", "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/data/2019-Nov.csv"]}
{"ts": "2026-03-16T10:46:51.090598+00:00", "level": "INFO", "event": "load_csvs_ok", "files": 2, "partitions": 114}
{"ts": "2026-03-16T10:46:51.091050+00:00", "level": "INFO", "event": "ingestion_completed", "elapsed_s": 0.09}
{"ts": "2026-03-16T10:46:51.091554+00:00", "level": "INFO", "event": "schema_valid", "columns": ["brand", "category_code", "category_id", "event_time", "event_type", "price", "product_id", "user_id", "user_session"]}


Partitions : 114
Columns    : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Schema valid : True


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,2103807459595387724,<NA>,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00+00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01+00:00,view,17200506,2053013559792632471,furniture.living_room.sofa,<NA>,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8


In [4]:
# ── Clean (lazy — still no compute) ──────────────────────────────────────────
with log.timer('cleaning') as t:
    clean_ddf = clean_data(raw_ddf)
timings['cleaning'] = t.elapsed

# ── Persist validated data as Parquet, partitioned by event_type ──────────────
# This is the first cell that triggers real computation:
#   read 14.6 GB CSV → filter → write Parquet. Expect 10-30 min on a laptop.
print('Writing validated Parquet — this triggers the first full read of the CSV files.')
print('Expected time: 10-30 minutes depending on disk speed.\n')
with log.timer('save_validated') as t:
    save_parquet_dask(clean_ddf, PARQUET_VALIDATED, partition_on=['event_type'])
timings['save_validated'] = t.elapsed

print(f'\nValidated Parquet → {PARQUET_VALIDATED}')
print(f'Elapsed           : {timings["save_validated"]:.1f}s')

# Reload from Parquet — all Dask analyses read this instead of the raw CSVs
validated_ddf = load_parquet(PARQUET_VALIDATED)
print(f'Reloaded partitions: {validated_ddf.npartitions}')

{"ts": "2026-03-16T10:46:53.738819+00:00", "level": "INFO", "event": "cleaning_started"}
{"ts": "2026-03-16T10:46:53.754328+00:00", "level": "INFO", "event": "cleaning_rules_applied"}
{"ts": "2026-03-16T10:46:53.754787+00:00", "level": "INFO", "event": "cleaning_completed", "elapsed_s": 0.02}
{"ts": "2026-03-16T10:46:53.755621+00:00", "level": "INFO", "event": "save_validated_started"}
{"ts": "2026-03-16T10:46:53.756158+00:00", "level": "INFO", "event": "saving_parquet_dask", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated"}


Writing validated Parquet — this triggers the first full read of the CSV files.
Expected time: 10-30 minutes depending on disk speed.



{"ts": "2026-03-16T10:51:29.861685+00:00", "level": "INFO", "event": "saved_parquet_dask", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated"}
{"ts": "2026-03-16T10:51:29.862598+00:00", "level": "INFO", "event": "save_validated_completed", "elapsed_s": 276.11}



Validated Parquet → /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated
Elapsed           : 276.1s
Reloaded partitions: 332


---
## Stage 2 — Distributed Processing with Dask

Five analyses, each using a distinct operation class:

| # | Analysis | Operations | Scheduler |
|---|---|---|---|
| 1 | Revenue by category | `filter` + `groupby` aggregate | distributed |
| 2 | Conversion funnel | per-event-type `filter` + single-key `groupby` | distributed |
| 3 | Hourly activity | per-event-type `filter` + `assign` + `groupby` | distributed |
| 4 | Session statistics | map-reduce `map_partitions` + pandas re-aggregation | **threaded** (no cluster) |
| 5 | Top brands | `filter` + `nlargest` | distributed |

> **Memory note:** Analysis 4 runs **outside** the distributed cluster because `groupby(user_session)` over millions of unique UUIDs triggers a `repartitiontofewer` step inside Dask's `.compute()` that OOMs workers. Without an active distributed client, the threaded scheduler concatenates partition results directly in the driver process.

In [ ]:
import dask

# Allow workers to spill to disk before hitting the kill threshold
dask.config.set({
    'distributed.worker.memory.target':    0.65,
    'distributed.worker.memory.spill':     0.75,
    'distributed.worker.memory.pause':     0.85,
    'distributed.worker.memory.terminate': 0.95,
})

dask_results = {}

# ── Analyses 1, 2, 3, 5 — run inside distributed cluster ────────────────────
with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='4GB') as cluster:
    with Client(cluster) as client:
        print(f'Dashboard : {client.dashboard_link}')
        print(f'Workers   : {len(client.scheduler_info()["workers"])}\n')

        # 1 — Revenue by category
        with log.timer('dask_revenue') as t:
            dask_results['revenue'] = compute_revenue_by_category(validated_ddf)
        timings['dask_revenue'] = t.elapsed
        print('--- Top 10 Categories by Revenue ---')
        print(dask_results['revenue'].head(10).to_string(index=False))

        # 2 — Conversion funnel
        with log.timer('dask_funnel') as t:
            dask_results['funnel'] = compute_conversion_funnel(validated_ddf)
        timings['dask_funnel'] = t.elapsed
        cols = [c for c in ['top_category','view','cart','purchase','purchase_rate','cart_rate']
                if c in dask_results['funnel'].columns]
        print('\n--- Conversion Funnel (top 5 by purchase rate) ---')
        print(dask_results['funnel'].sort_values('purchase_rate', ascending=False).head(5)[cols].to_string(index=False))

        # 3 — Hourly activity
        with log.timer('dask_hourly') as t:
            dask_results['hourly'] = compute_hourly_activity(validated_ddf)
        timings['dask_hourly'] = t.elapsed
        print('\n--- Events by Hour (purchases) ---')
        hourly_purchase = dask_results['hourly'][dask_results['hourly']['event_type'] == 'purchase']
        for _, row in hourly_purchase.iterrows():
            bar = '#' * (int(row['count']) // 50_000)
            print(f'  {int(row["hour"]):2d}:00  {int(row["count"]):>10,}  {bar}')

        # 5 — Top brands
        with log.timer('dask_brands') as t:
            dask_results['brands'] = compute_top_brands(validated_ddf, top_n=20)
        timings['dask_brands'] = t.elapsed
        print('\n--- Top 10 Brands by Purchase Revenue ---')
        print(dask_results['brands'].head(10).to_string(index=False))

# ── Analysis 4: Session stats — no active distributed client ─────────────────
print('\n--- Session Analysis (threaded scheduler, no distributed client) ---')
with log.timer('dask_sessions') as t:
    dask_results['sessions'] = compute_session_stats(validated_ddf)
timings['dask_sessions'] = t.elapsed
sess = dask_results['sessions']
print(f'  Total sessions      : {len(sess):,}')
print(f'  Avg events/session  : {sess["num_events"].mean().compute():.1f}')
print(f'  Median duration     : {sess["session_duration_min"].median().compute():.1f} min')
print(f'  Avg spend/session   : {sess["total_spend"].mean().compute():.2f}')

print(f'\nDask compute total: {sum(v for k,v in timings.items() if k.startswith("dask_")):.1f}s')

{"ts": "2026-03-16T10:51:30.766009+00:00", "level": "INFO", "event": "dask_revenue_started"}


Dashboard : http://127.0.0.1:8787/status
Workers   : 2



2026-03-16 11:51:31,570 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27a22f690b7b652c3206c305ae156289 initialized by task ('shuffle-transfer-27a22f690b7b652c3206c305ae156289', 36) executed on worker tcp://127.0.0.1:46167
2026-03-16 11:51:40,677 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27a22f690b7b652c3206c305ae156289 deactivated due to stimulus 'task-finished-1773658300.6739993'
{"ts": "2026-03-16T10:51:40.730427+00:00", "level": "INFO", "event": "revenue_by_category_done", "categories": 14}
{"ts": "2026-03-16T10:51:40.731035+00:00", "level": "INFO", "event": "dask_revenue_completed", "elapsed_s": 9.97}
{"ts": "2026-03-16T10:51:40.733702+00:00", "level": "INFO", "event": "dask_funnel_started"}


--- Top 10 Categories by Revenue ---
top_category  total_revenue  num_purchases  avg_price
 electronics   381714286.57         916667 416.415434
     unknown    52805443.81         407643 129.538454
  appliances    32223623.59         174022 185.169827
   computers    25373205.34          62332 407.065477
   furniture     4216980.01          19843 212.517261
        auto     2649036.47          21339 124.140610
construction     2013385.72          16500 122.023377
     apparel     1805962.87          22217  81.287432
        kids     1401903.93          11648 120.355763
       sport      718251.75           2725 263.578624


2026-03-16 11:51:41,454 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5813704845ca20e5f62f83924a2afa94 initialized by task ('shuffle-transfer-5813704845ca20e5f62f83924a2afa94', 37) executed on worker tcp://127.0.0.1:41953
2026-03-16 11:52:48,602 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5813704845ca20e5f62f83924a2afa94 deactivated due to stimulus 'task-finished-1773658368.6007957'
2026-03-16 11:52:49,147 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 393e904ba5bc743d0dd44e54b14531ac initialized by task ('shuffle-transfer-393e904ba5bc743d0dd44e54b14531ac', 40) executed on worker tcp://127.0.0.1:46167
2026-03-16 11:53:59,401 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 393e904ba5bc743d0dd44e54b14531ac deactivated due to stimulus 'task-finished-1773658439.3998325'
2026-03-16 11:54:00,212 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f1254fabb4b55875a0c75f02bf89212d initialized by task ('shuffle-transfer-f1254fabb4b5

...


--- Events by Hour (purchases) ---
   0:00       5,771  
   1:00       9,741  
   2:00      25,124  
   3:00      55,970  #
   4:00      85,235  #
   5:00     101,432  ##
   6:00     109,432  ##
   7:00     112,111  ##
   8:00     120,458  ##
   9:00     126,617  ##
  10:00     120,946  ##
  11:00     111,582  ##
  12:00     103,145  ##
  13:00      98,809  #
  14:00      96,837  #
  15:00      90,231  #
  16:00      81,993  #
  17:00      76,215  #
  18:00      47,967  
  19:00      33,811  
  20:00      20,045  
  21:00      12,597  
  22:00       8,126  
  23:00       5,593  


2026-03-16 11:56:59,139 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b4c7c99061396fdffb70ee41f52be3e7 initialized by task ('shuffle-transfer-b4c7c99061396fdffb70ee41f52be3e7', 42) executed on worker tcp://127.0.0.1:41953
2026-03-16 11:57:07,706 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b4c7c99061396fdffb70ee41f52be3e7 deactivated due to stimulus 'task-finished-1773658627.7029986'
{"ts": "2026-03-16T10:57:07.744811+00:00", "level": "INFO", "event": "top_brands_done", "brands": 20}
{"ts": "2026-03-16T10:57:07.754457+00:00", "level": "INFO", "event": "dask_brands_completed", "elapsed_s": 8.86}



--- Top 10 Brands by Purchase Revenue ---
  brand  total_revenue  num_purchases
  apple   238721793.70         308937
samsung   101277413.48         372923
 xiaomi    20453899.25         124908
 huawei     9664104.09          47204
     lg     8626906.72          21606
   acer     6924026.05          13284
lucente     6651658.94          26137
   sony     6341082.98          17038
   oppo     5901500.52          25971
 lenovo     4450744.83          11125


{"ts": "2026-03-16T10:57:08.479228+00:00", "level": "INFO", "event": "dask_sessions_started"}
{"ts": "2026-03-16T10:57:08.479839+00:00", "level": "INFO", "event": "session_stats_started"}
{"ts": "2026-03-16T10:57:08.505576+00:00", "level": "INFO", "event": "dask_sessions_completed", "elapsed_s": 0.03}



--- Session Analysis (threaded scheduler, no distributed client) ---


In [ ]:
# ── Persist Dask results as Parquet ───────────────────────────────────────────
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_parquet_pandas(dask_results['revenue'],                RESULTS_DIR / 'dask_revenue.parquet')
save_parquet_pandas(dask_results['funnel'],                 RESULTS_DIR / 'dask_funnel.parquet')
save_parquet_pandas(dask_results['hourly'],                 RESULTS_DIR / 'dask_hourly.parquet')
save_parquet_pandas(dask_results['brands'],                 RESULTS_DIR / 'dask_brands.parquet')
save_parquet_pandas(dask_results['sessions'].head(500_000), RESULTS_DIR / 'dask_sessions.parquet')

print('Dask results saved to:', RESULTS_DIR)

{"ts": "2026-03-16T10:31:43.843291+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_revenue.parquet", "rows": 0}
{"ts": "2026-03-16T10:31:43.845315+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_funnel.parquet", "rows": 11}
{"ts": "2026-03-16T10:31:43.846872+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_hourly.parquet", "rows": 16}
{"ts": "2026-03-16T10:31:43.848225+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_brands.parquet", "rows": 0}
{"ts": "2026-03-16T10:31:43.905190+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/

Dask results saved to: /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results


---
## Stage 3 — Distributed Processing with Apache Spark

Same analytical questions answered via PySpark for framework comparison.  
Spark reads **directly from the raw CSV files** to avoid Parquet timestamp format incompatibilities between pyarrow (Dask) and the JVM Parquet reader (Spark).  
**Bonus**: analysis 3 uses a **window function** (`RANK() OVER PARTITION BY`) to rank products within each category.

> If PySpark is not installed, this stage is skipped automatically.

In [ ]:
spark_results = {}

if not SPARK_AVAILABLE:
    print('Spark not available — skipping Stage 3.')
else:
    from pyspark.sql import functions as F

    spark = get_spark_session('EcomPipeline-A7')
    print(f'Spark version: {spark.version}\n')

    # Spark reads the raw CSVs directly — avoids pyarrow/JVM Parquet incompatibility
    # Use forward-slash paths (Spark on Windows requires this)
    csv_paths = [
        str(RAW_OCT).replace('\\', '/'),
        str(RAW_NOV).replace('\\', '/'),
    ]
    with log.timer('spark_load') as t:
        sdf_raw = load_csv_spark(spark, csv_paths)
    timings['spark_load'] = t.elapsed

    # Apply the same cleaning filters as Stage 1
    valid_events = list(VALID_EVENT_TYPES)
    sdf = (
        sdf_raw
        .dropna(subset=['user_id', 'product_id', 'price', 'event_time', 'user_session'])
        .filter(F.col('event_type').isin(valid_events))
        .filter(F.col('price') >= 0)
        .filter(F.trim(F.col('user_session')) != '')
    )
    print(f'Spark partitions after cleaning: {sdf.rdd.getNumPartitions()}')
    sdf.printSchema()

    # 1 — Revenue by category
    with log.timer('spark_revenue') as t:
        spark_results['revenue'] = compute_revenue_by_category_spark(sdf)
    timings['spark_revenue'] = t.elapsed
    print('--- Spark: Top 10 Categories by Revenue ---')
    spark_results['revenue'].show(10, truncate=False)

    # 2 — Conversion funnel
    with log.timer('spark_funnel') as t:
        spark_results['funnel'] = compute_conversion_funnel_spark(sdf)
    timings['spark_funnel'] = t.elapsed
    print('--- Spark: Conversion Funnel ---')
    spark_results['funnel'].orderBy('purchase_rate', ascending=False).show(10, truncate=False)

    # 3 — Window function: rank products within category
    with log.timer('spark_window') as t:
        spark_results['window'] = compute_window_rank_spark(sdf)
    timings['spark_window'] = t.elapsed
    print('--- Spark: Top Products per Category (window rank) ---')
    spark_results['window'].show(20, truncate=False)

    # 4 — Hourly activity
    with log.timer('spark_hourly') as t:
        spark_results['hourly'] = compute_hourly_activity_spark(sdf)
    timings['spark_hourly'] = t.elapsed

    # Persist Spark results as Parquet
    results_fwd = str(RESULTS_DIR).replace('\\', '/')
    spark_results['revenue'].write.mode('overwrite').parquet(f'{results_fwd}/spark_revenue')
    spark_results['funnel'].write.mode('overwrite').parquet(f'{results_fwd}/spark_funnel')
    spark_results['window'].write.mode('overwrite').parquet(f'{results_fwd}/spark_window_rank')
    spark_results['hourly'].write.mode('overwrite').parquet(f'{results_fwd}/spark_hourly')

    print(f'\nSpark compute total: {sum(v for k,v in timings.items() if k.startswith("spark_")):.1f}s')
    spark.stop()

{"ts": "2026-03-16T10:31:43.969398+00:00", "level": "INFO", "event": "spark_load_started"}


Spark version: 4.1.1



{"ts": "2026-03-16T10:32:28.440590+00:00", "level": "INFO", "event": "test_mode_enabled", "action": "limiting_spark_input"}
{"ts": "2026-03-16T10:32:28.482305+00:00", "level": "INFO", "event": "spark_csv_loaded", "files": 2, "partitions": 1}
{"ts": "2026-03-16T10:32:28.482748+00:00", "level": "INFO", "event": "spark_load_completed", "elapsed_s": 44.51}
{"ts": "2026-03-16T10:32:35.479237+00:00", "level": "INFO", "event": "spark_revenue_started"}
{"ts": "2026-03-16T10:32:35.531698+00:00", "level": "INFO", "event": "spark_revenue_done"}
{"ts": "2026-03-16T10:32:35.532164+00:00", "level": "INFO", "event": "spark_revenue_completed", "elapsed_s": 0.05}


Spark partitions after cleaning: 1
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

--- Spark: Top 10 Categories by Revenue ---


{"ts": "2026-03-16T10:32:41.767886+00:00", "level": "INFO", "event": "spark_funnel_started"}


...

{"ts": "2026-03-16T10:32:54.753272+00:00", "level": "INFO", "event": "spark_window_started"}
{"ts": "2026-03-16T10:32:54.792959+00:00", "level": "INFO", "event": "spark_window_rank_done"}
{"ts": "2026-03-16T10:32:54.793461+00:00", "level": "INFO", "event": "spark_window_completed", "elapsed_s": 0.04}


+------------+----+--------+-----+---------------------+--------------------+
|top_category|cart|purchase|view |purchase_rate        |cart_rate           |
+------------+----+--------+-----+---------------------+--------------------+
|stationery  |1   |2       |28   |0.07142857142857142  |0.03571428571428571 |
|medicine    |0   |2       |36   |0.05555555555555555  |0.0                 |
|electronics |1022|925     |35014|0.026418004226880676 |0.029188324670131948|
|appliances  |38  |177     |12159|0.014557118184061189 |0.003125257011267374|
|computers   |14  |72      |5508 |0.013071895424836602 |0.002541757443718228|
|accessories |0   |8       |630  |0.012698412698412698 |0.0                 |
|unknown     |125 |390     |32072|0.012160139685707158 |0.003897480668495884|
|kids        |0   |12      |1184 |0.010135135135135136 |0.0                 |
|furniture   |2   |21      |2625 |0.008                |7.619047619047619E-4|
|construction|9   |14      |1945 |0.0071979434447300775|0.004627

{"ts": "2026-03-16T10:33:01.107281+00:00", "level": "INFO", "event": "spark_hourly_started"}
{"ts": "2026-03-16T10:33:01.122892+00:00", "level": "INFO", "event": "spark_hourly_done"}
{"ts": "2026-03-16T10:33:01.123423+00:00", "level": "INFO", "event": "spark_hourly_completed", "elapsed_s": 0.02}


+------------+----------+---------------+---------+----------------+
|top_category|product_id|product_revenue|num_sales|rank_in_category|
+------------+----------+---------------+---------+----------------+
|accessories |28401176  |602.34         |6        |1               |
|accessories |28300432  |46.33          |1        |2               |
|accessories |18300229  |42.64          |1        |3               |
|apparel     |28719130  |151.61         |1        |1               |
|apparel     |28719171  |137.71         |1        |2               |
|apparel     |28717864  |122.27         |1        |3               |
|apparel     |28715839  |102.44         |2        |4               |
|apparel     |28717073  |100.13         |1        |5               |
|appliances  |5000401   |1611.22        |1        |1               |
|appliances  |3601489   |1428.54        |3        |2               |
|appliances  |4600603   |1368.88        |2        |3               |
|appliances  |3600661   |1181.96  


Spark compute total: 51.2s


---
## Stage 4 — Results Summary & Framework Comparison

### Storage Strategy

| Layer | Format | Location | Why |
|---|---|---|---|
| Raw input | CSV | `data/` | Source of truth, unchanged |
| Validated intermediate | **Parquet** (partitioned by `event_type`) | `output/parquet/validated/` | Predicate pushdown speeds downstream reads |
| Analysis results | **Parquet** (single file) | `output/results/` | Compact, typed, directly loadable by BI tools |

In [ ]:
# ── Timing comparison ─────────────────────────────────────────────────────────
print('=== Pipeline Stage Timings ===')
print(f'{"Stage":<25} {"Time (s)":>10}')
print('-' * 37)
for stage, elapsed in timings.items():
    print(f'{stage:<25} {elapsed:>10.1f}')
print('=' * 37)
print(f'{"TOTAL":<25} {sum(timings.values()):>10.1f}')

# ── Output files ──────────────────────────────────────────────────────────────
print('\n=== Output Files ===')
for p in sorted(Path(OUTPUT_DIR).rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1_048_576
        print(f'  {str(p.relative_to(PROJECT_ROOT)):<55} {size_mb:>7.2f} MB')

# ── Framework notes ───────────────────────────────────────────────────────────
print("""
=== Framework Comparison Notes ===
Dask
  + Familiar pandas-like API, easy iteration in notebooks
  + Native Python, no JVM overhead
  + Low-latency scheduler startup
  - High-cardinality groupby OOMs via repartitiontofewer in distributed mode;
    must run outside Client context with threaded scheduler
  - Multi-key shuffles require per-key decomposition to avoid OOM

Spark
  + Catalyst optimiser produces efficient physical plans
  + First-class window functions, complex joins
  + Graceful spill-to-disk for large shuffles
  + No format interoperability issues (reads CSV directly)
  - JVM startup adds ~30s latency
  - More verbose API; Windows path handling quirks

Verdict: Dask is faster for simple aggregations on a single node.
Spark is more robust for complex operations (windows, multi-key joins)
and scales better on managed clusters (EMR, Dataproc).
""")

=== Pipeline Stage Timings ===
Stage                       Time (s)
-------------------------------------
ingestion                        0.0
cleaning                         0.0
save_validated                   2.9
dask_revenue                     0.5
dask_funnel                      0.5
dask_hourly                      1.0
dask_brands                      0.1
dask_sessions                    0.0
spark_load                      44.5
spark_revenue                    0.1
spark_funnel                     6.6
spark_window                     0.0
spark_hourly                     0.0
TOTAL                           56.3

=== Output Files ===
  output/parquet/validated/event_type=cart/part.0.parquet    0.55 MB
  output/parquet/validated/event_type=cart/part.1.parquet    0.55 MB
  output/parquet/validated/event_type=cart/part.10.parquet    0.47 MB
  output/parquet/validated/event_type=cart/part.100.parquet    1.74 MB
  output/parquet/validated/event_type=cart/part.101.parquet    1.36 MB
  ou